In [1]:
import pandas as pd
import numpy as np
import fastparquet

In [2]:
####################################################################################################################
# Carrega Sinan
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

In [3]:
# Ajusta variaveis do SINAN

# Padroniza codigo municipio do Sinan como numero inteiro
sinan['ID_MN_RESI'] = sinan['ID_MN_RESI'].astype(int)

# Sinan para construir variavel MG
sinan_mg = sinan

# Converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    sinan_mg[col] = pd.to_numeric(sinan_mg[col], errors="coerce").fillna(0)

In [4]:

####################################################################################################################
# Cria variavel Moderado/Grave (MG)


# =========================================================
# FILTRO DE CASOS MODERADOS/GRAVES QUALIFICADOS
# ESCORPIONISMO
# =========================================================


# ---------------------------------------------------------
# CRITÉRIOS FORTES
# Isoladamente já sugerem fortemente MG
# ---------------------------------------------------------

criterio_forte = (

    # SAA >= 2 ampolas
    (sinan_mg["NU_AMPOL_8"] >= 2) |

    # SAEsc >= 2 ampolas
    (sinan_mg["NU_AMPOL_9"] >= 2) |

    # Óbito por animais peçonhentos
    (sinan_mg["EVOLUCAO"] == "Obito por ap") |

    # Manifestações vagais
    (sinan_mg["CLI_VAGAIS"] == "Sim")

)

# ---------------------------------------------------------
# CRITÉRIOS ASSOCIATIVOS
# Variáveis sujeitas a erro de preenchimento,
# mas que em conjunto aumentam a probabilidade
# de representar MG
# ---------------------------------------------------------

criterio_associativo = (

    # Soroterapia + classificação moderado/grave
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"]))
    ) |

    # Soroterapia + manifestações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["MCLI_SIST"] == "Sim")
    ) |

    # Soroterapia + complicações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["COM_SISTEM"] == "Sim")
    )

)

# ---------------------------------------------------------
# FILTRO FINAL
# ---------------------------------------------------------

filtro_mg = criterio_forte | criterio_associativo


# ---------------------------------------------------------
# CRIAR VARIÁVEL BINÁRIA
# ---------------------------------------------------------

sinan_mg["MG"] = np.where(filtro_mg, 1, 0)

# ---------------------------------------------------------
# CONFERÊNCIA
# ---------------------------------------------------------

print("Total de casos:", len(sinan_mg))
print("Moderados/Graves qualificados:", filtro_mg.sum())
print("Proporção:", round(filtro_mg.mean() * 100, 2), "%")


Total de casos: 117357
Moderados/Graves qualificados: 4475
Proporção: 3.81 %


In [ ]:
# Cria colunas
# Total de casos (sem distincao por faixa etaria)
total_casos = sinan_mg['ID_MN_RESI'].value_counts().reset_index(name='TOTAL_CASOS')

# Moderados/graves (total por municipio)
total_mg = sinan_mg.pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

total_mg.columns = ['ID_MN_RESI','LEVE','MG'] # Renomeia as colunas

In [6]:
#df.to_excel('Dados-iniciais/base02.xlsx')